# Credit Risk Modeling — SageMaker Training Notebook

**Pipeline**: Bronze → Silver → **Gold → Modeling (notebook ini)**

**Input** : `s3://credit-risk-checker-datalake-*/03-gold/train_features_gold/`  
**Target** : `TARGET` (1 = default, 0 = tidak default)  
**Model** : 4 model — Logistic Regression (baseline), Random Forest, XGBoost, LightGBM  
**Metrik utama** : ROC-AUC (standar industri credit scoring)

---

## 0. Setup & Install Dependencies

Jalankan cell instalasi, **restart kernel**, lalu lanjutkan ke cell import.

In [ ]:
%pip install -q --upgrade \
    'pyarrow>=15.0.0' \
    'pandas>=2.1.0' \
    'scikit-learn>=1.4.0' \
    'xgboost>=2.0.0' \
    'lightgbm>=4.3.0' \
    'imbalanced-learn>=0.12.0' \
    's3fs>=2024.2.0' \
    'fsspec>=2024.2.0' \
    'matplotlib>=3.7.0' \
    'seaborn>=0.12.0' \
    'shap>=0.46.0'

In [ ]:
import boto3
import sagemaker
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings
warnings.filterwarnings('ignore')

# SageMaker session
session = sagemaker.Session()
role    = sagemaker.get_execution_role()
region  = boto3.Session().region_name

print(f"Region  : {region}")
print(f"Role    : {role}")
print(f"Bucket  : {session.default_bucket()}")

## 1. Load Data dari Gold Layer (S3)

In [ ]:
# ── KONFIGURASI PATH S3 ────────────────────────────────────────────────────────
BUCKET     = "credit-risk-checker-datalake-242046727318-us-east-1-an"
GOLD_TRAIN = f"03-gold/train_features_gold/"
GOLD_TEST  = f"03-gold/test_features_gold/"

# ── BACA PARQUET VIA PYARROW + S3 (hindari konflik pandas extension type) ─────
# Menggunakan pyarrow.dataset untuk baca folder parquet dari S3
import pyarrow.dataset as ds
import pyarrow as pa
import s3fs

fs = s3fs.S3FileSystem()  # pakai IAM role SageMaker, tidak perlu credentials manual

def read_parquet_folder(bucket, prefix):
    """Baca semua file parquet dalam folder S3 menggunakan pyarrow.dataset.
    Lebih aman dari pd.read_parquet karena hindari registrasi extension type ulang."""
    dataset = ds.dataset(f"{bucket}/{prefix}", filesystem=fs, format='parquet')
    table   = dataset.to_table()
    return table.to_pandas()

df_train = read_parquet_folder(BUCKET, GOLD_TRAIN)
df_test  = read_parquet_folder(BUCKET, GOLD_TEST)

print(f"Train shape : {df_train.shape}")
print(f"Test shape  : {df_test.shape}")
print(f"\nClass distribution (TARGET):")
print(df_train['TARGET'].value_counts(normalize=True).rename({0: 'Non-default (0)', 1: 'Default (1)'}).to_string())

In [ ]:
# Preview kolom gold
print(f"Total kolom: {len(df_train.columns)}")
print("\nKolom:")
for col in df_train.columns:
    print(f"  {col:45s} | dtype: {str(df_train[col].dtype):10s} | null%: {df_train[col].isna().mean():.1%}")

## 2. Eksplorasi Data (EDA Ringkas)

In [ ]:
# ── DISTRIBUSI KELAS ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Class balance
class_counts = df_train['TARGET'].value_counts()
axes[0].bar(['Non-default (0)', 'Default (1)'], class_counts.values, color=['#2196F3', '#F44336'])
axes[0].set_title('Distribusi Target (Class Balance)')
axes[0].set_ylabel('Jumlah Nasabah')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, f'{v:,}\n({v/len(df_train):.1%})', ha='center')

# Debt-to-income by target
df_train.groupby('TARGET')['debt_to_income'].median().plot(kind='bar', ax=axes[1], color=['#2196F3', '#F44336'])
axes[1].set_title('Median Debt-to-Income by Target')
axes[1].set_ylabel('Debt-to-Income Ratio')
axes[1].set_xticklabels(['Non-default (0)', 'Default (1)'], rotation=0)

plt.tight_layout()
plt.savefig('/tmp/eda_overview.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── KORELASI FITUR NUMERIK DENGAN TARGET ─────────────────────────────────────
num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c not in ['SK_ID_CURR', 'TARGET']]

correlations = df_train[num_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
top_corr = correlations.abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
colors = ['#F44336' if correlations[c] > 0 else '#2196F3' for c in top_corr.index]
plt.barh(top_corr.index[::-1], [correlations[c] for c in top_corr.index[::-1]], color=colors[::-1])
plt.xlabel('Korelasi dengan TARGET')
plt.title('Top 20 Fitur — Korelasi dengan Default (TARGET=1)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('/tmp/feature_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Preprocessing & Feature Preparation

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, confusion_matrix, ConfusionMatrixDisplay
)

# ── PISAH FITUR & TARGET ──────────────────────────────────────────────────────
EXCLUDE_COLS = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df_train.columns if c not in EXCLUDE_COLS]

X = df_train[feature_cols].copy()
y = df_train['TARGET'].copy()

# ── KATEGORIKAN KOLOM ─────────────────────────────────────────────────────────
CAT_COLS = X.select_dtypes(include=['object', 'category']).columns.tolist()
NUM_COLS = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Total fitur   : {len(feature_cols)}")
print(f"Fitur numerik : {len(NUM_COLS)}")
print(f"Fitur kategori: {len(CAT_COLS)}")
print(f"\nKolom kategori: {CAT_COLS}")

In [ ]:
# ── ENCODE FITUR KATEGORI (Label Encoding untuk tree-based, OHE untuk LR) ────
# Strategi: buat versi label-encoded (untuk RF/XGB/LGBM) dan ordinal (untuk LR)

X_enc = X.copy()
label_encoders = {}

for col in CAT_COLS:
    le = LabelEncoder()
    X_enc[col] = X_enc[col].astype(str).fillna('__missing__')
    X_enc[col] = le.fit_transform(X_enc[col])
    label_encoders[col] = le

# ── TRAIN / VALIDATION SPLIT ──────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_enc, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}  |  X_val: {X_val.shape}")
print(f"y_train default rate: {y_train.mean():.2%}")
print(f"y_val   default rate: {y_val.mean():.2%}")

In [ ]:
# ── IMBALANCED CLASS HANDLING ─────────────────────────────────────────────────
# Dataset ini biasanya ~8% default → pakai class_weight='balanced' atau scale_pos_weight
class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Rasio kelas (0:1) = {class_ratio:.1f}x")
print("→ Semua model akan menggunakan class_weight='balanced' atau scale_pos_weight")

## 4. Model 1 — Logistic Regression (Baseline)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression butuh scaling dan imputation yang clean
imputer = SimpleImputer(strategy='median')
scaler  = StandardScaler()

X_train_lr = scaler.fit_transform(imputer.fit_transform(X_train))
X_val_lr   = scaler.transform(imputer.transform(X_val))

lr_model = LogisticRegression(
    C=0.1,                  # regularisasi L2
    class_weight='balanced',
    solver='saga',          # efisien untuk dataset besar
    max_iter=500,
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train_lr, y_train)

lr_pred_proba = lr_model.predict_proba(X_val_lr)[:, 1]
lr_auc  = roc_auc_score(y_val, lr_pred_proba)
lr_aucpr = average_precision_score(y_val, lr_pred_proba)

print(f"[Logistic Regression] ROC-AUC: {lr_auc:.4f} | PR-AUC: {lr_aucpr:.4f}")

## 5. Model 2 — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest tidak butuh scaling; handle missing dengan SimpleImputer
imputer_rf = SimpleImputer(strategy='median')
X_train_rf = imputer_rf.fit_transform(X_train)
X_val_rf   = imputer_rf.transform(X_val)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=50,    # cegah overfitting pada kelas minoritas
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train_rf, y_train)

rf_pred_proba = rf_model.predict_proba(X_val_rf)[:, 1]
rf_auc  = roc_auc_score(y_val, rf_pred_proba)
rf_aucpr = average_precision_score(y_val, rf_pred_proba)

print(f"[Random Forest] ROC-AUC: {rf_auc:.4f} | PR-AUC: {rf_aucpr:.4f}")

## 6. Model 3 — XGBoost

In [ ]:
import xgboost as xgb

# XGBoost handle missing natively; tidak butuh imputer
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=50,
    scale_pos_weight=class_ratio,   # handle imbalanced
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
    tree_method='hist',             # lebih cepat di SageMaker
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

xgb_pred_proba = xgb_model.predict_proba(X_val)[:, 1]
xgb_auc  = roc_auc_score(y_val, xgb_pred_proba)
xgb_aucpr = average_precision_score(y_val, xgb_pred_proba)

print(f"\n[XGBoost] ROC-AUC: {xgb_auc:.4f} | PR-AUC: {xgb_aucpr:.4f}")

## 7. Model 4 — LightGBM

In [ ]:
import lightgbm as lgb

# LightGBM juga handle missing natively
lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
    lgb.log_evaluation(period=100)
]

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=callbacks
)

lgb_pred_proba = lgb_model.predict_proba(X_val)[:, 1]
lgb_auc  = roc_auc_score(y_val, lgb_pred_proba)
lgb_aucpr = average_precision_score(y_val, lgb_pred_proba)

print(f"\n[LightGBM] ROC-AUC: {lgb_auc:.4f} | PR-AUC: {lgb_aucpr:.4f}")

## 8. Perbandingan Model

In [ ]:
# ── TABEL RINGKASAN METRIK ────────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': [
        'Logistic Regression (baseline)',
        'Random Forest',
        'XGBoost',
        'LightGBM'
    ],
    'ROC-AUC': [lr_auc, rf_auc, xgb_auc, lgb_auc],
    'PR-AUC' : [lr_aucpr, rf_aucpr, xgb_aucpr, lgb_aucpr],
})
results = results.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results['ROC-AUC'] = results['ROC-AUC'].map('{:.4f}'.format)
results['PR-AUC']  = results['PR-AUC'].map('{:.4f}'.format)
print("\n=== RINGKASAN PERFORMA MODEL ===")
print(results.to_string(index=False))

In [ ]:
# ── ROC CURVE SEMUA MODEL ─────────────────────────────────────────────────────
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC Curve
models_info = [
    ('Logistic Regression', lr_pred_proba,  lr_auc,  '#9E9E9E'),
    ('Random Forest',       rf_pred_proba,  rf_auc,  '#2196F3'),
    ('XGBoost',             xgb_pred_proba, xgb_auc, '#FF9800'),
    ('LightGBM',            lgb_pred_proba, lgb_auc, '#4CAF50'),
]

for name, proba, auc, color in models_info:
    fpr, tpr, _ = roc_curve(y_val, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, linewidth=2)

axes[0].plot([0,1],[0,1], 'k--', alpha=0.4, label='Random (AUC=0.50)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Perbandingan Model')
axes[0].legend(loc='lower right')
axes[0].grid(alpha=0.3)

# PR Curve
from sklearn.metrics import precision_recall_curve

for name, proba, _, color in models_info:
    precision, recall, _ = precision_recall_curve(y_val, proba)
    ap = average_precision_score(y_val, proba)
    axes[1].plot(recall, precision, label=f'{name} (AP={ap:.4f})', color=color, linewidth=2)

baseline_pr = y_val.mean()
axes[1].axhline(baseline_pr, color='k', linestyle='--', alpha=0.4, label=f'Baseline (AP={baseline_pr:.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — Perbandingan Model')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/roc_pr_curves.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── CONFUSION MATRIX (threshold = 0.5) ───────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, proba, _, color) in zip(axes, models_info):
    pred = (proba >= 0.5).astype(int)
    cm   = confusion_matrix(y_val, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Non-default', 'Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=10)

plt.suptitle('Confusion Matrix (threshold=0.5)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

# Classification report model terbaik
best_proba = xgb_pred_proba  # ganti jika model lain lebih baik
print("\n=== Classification Report (XGBoost, threshold=0.5) ===")
print(classification_report(y_val, (best_proba >= 0.5).astype(int),
                             target_names=['Non-default', 'Default']))

## 9. Feature Importance

In [ ]:
# ── FEATURE IMPORTANCE LGBM & XGB ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def plot_importance(importances, feature_names, title, ax, color, top_n=25):
    fi = pd.Series(importances, index=feature_names).sort_values(ascending=True).tail(top_n)
    fi.plot(kind='barh', ax=ax, color=color)
    ax.set_title(title)
    ax.set_xlabel('Importance')

plot_importance(
    lgb_model.feature_importances_,
    X_train.columns,
    'LightGBM — Top 25 Feature Importance',
    axes[0], '#4CAF50'
)

plot_importance(
    xgb_model.feature_importances_,
    X_train.columns,
    'XGBoost — Top 25 Feature Importance',
    axes[1], '#FF9800'
)

plt.tight_layout()
plt.savefig('/tmp/feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── LOGISTIC REGRESSION COEFFICIENTS ─────────────────────────────────────────
lr_coef = pd.Series(lr_model.coef_[0], index=X_train.columns)
top_lr  = lr_coef.abs().sort_values(ascending=True).tail(20)

plt.figure(figsize=(10, 6))
colors = ['#F44336' if lr_coef[c] > 0 else '#2196F3' for c in top_lr.index]
plt.barh(top_lr.index, [lr_coef[c] for c in top_lr.index], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (merah = tingkatkan risiko, biru = turunkan risiko)')
plt.title('Logistic Regression — Top 20 Coefficients')
plt.tight_layout()
plt.savefig('/tmp/lr_coefficients.png', dpi=100, bbox_inches='tight')
plt.show()

## 10. Tuning Threshold — Business Decision

In [ ]:
# ── THRESHOLD ANALYSIS ────────────────────────────────────────────────────────
# Dalam credit risk, kita ingin minimize false negative (default yang lolos)
# Threshold lebih rendah → recall Default lebih tinggi, precision lebih rendah

thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []

for t in thresholds:
    pred = (xgb_pred_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    recall_default    = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_default = tp / (tp + fp) if (tp + fp) > 0 else 0
    approval_rate     = 1 - pred.mean()  # proporsi yang disetujui
    threshold_results.append({
        'Threshold'       : round(t, 2),
        'Recall Default'  : round(recall_default, 3),
        'Precision Default': round(precision_default, 3),
        'Approval Rate'   : round(approval_rate, 3),
    })

df_thresh = pd.DataFrame(threshold_results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_thresh['Threshold'], df_thresh['Recall Default'],    label='Recall Default',    color='#F44336', marker='o')
ax.plot(df_thresh['Threshold'], df_thresh['Precision Default'], label='Precision Default', color='#FF9800', marker='s')
ax.plot(df_thresh['Threshold'], df_thresh['Approval Rate'],     label='Approval Rate',     color='#2196F3', marker='^')
ax.axvline(0.3, color='green', linestyle='--', alpha=0.6, label='Threshold rekomendasi (0.3)')
ax.set_xlabel('Threshold')
ax.set_ylabel('Rate')
ax.set_title('Trade-off Threshold — XGBoost (Business Decision)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/threshold_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetail per threshold:")
print(df_thresh.to_string(index=False))

## 11. Cross-Validation (Robustness Check)

In [ ]:
# ── 5-FOLD STRATIFIED CV — MODEL TERBAIK SAJA (untuk efisiensi) ───────────────
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Catatan: early_stopping_rounds TIDAK dipakai di sini karena cross_val_score
# tidak menyediakan eval_set per fold (early stopping butuh validation set eksplisit)
xgb_cv = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=50,
    scale_pos_weight=class_ratio,
    eval_metric='auc',
    random_state=42,
    tree_method='hist',
    n_jobs=-1,
    verbosity=0
)

cv_scores = cross_val_score(
    xgb_cv, X_enc, y,
    cv=cv, scoring='roc_auc',
    n_jobs=-1
)

print(f"[XGBoost 5-Fold CV] ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  Fold scores: {[f'{s:.4f}' for s in cv_scores]}")

## 11.5 Model Explainability — SHAP (SHapley Additive exPlanations)

`feature_importances_` (Bagian 9) hanya memberi ranking **global** dan tidak menunjukkan **arah** pengaruh (apakah nilai fitur yang tinggi menaikkan atau menurunkan risiko) maupun kontribusi untuk **satu nasabah spesifik**. SHAP mengatasi keterbatasan ini dengan menghitung kontribusi tiap fitur terhadap setiap prediksi berdasarkan teori Shapley value (game theory), sehingga konsisten secara matematis dan additive (jumlah kontribusi semua fitur + base value = output model).

Dua manfaat utama di credit risk modeling:
- **Global** — fitur apa saja yang paling mendorong risiko default secara keseluruhan, dan ke arah mana.
- **Lokal** — untuk satu nasabah tertentu, fitur apa yang membuat skornya naik/turun. Ini penting untuk transparansi keputusan kredit (mis. *adverse action notice* / alasan penolakan ke nasabah, dan audit trail untuk regulator seperti OJK).

In [ ]:
# ── INSTALL SHAP (jika belum terpasang dari Cell 1) ──────────────────────────
import sys
!{sys.executable} -m pip install -q shap==0.46.0
import shap
print(f"SHAP version: {shap.__version__}")

### 11.5.1 Hitung SHAP Values

`TreeExplainer` eksak (bukan aproksimasi) dan cepat untuk model tree-based (Random Forest, XGBoost, LightGBM). Di sini kita pakai **XGBoost** (`xgb_model`) karena model ini dipakai sebagai acuan utama di bagian-bagian sebelumnya (confusion matrix, threshold tuning, CV) — tinggal ganti `SHAP_MODEL` ke `lgb_model` atau `rf_model` untuk explain model lain. Untuk Logistic Regression, gunakan `shap.LinearExplainer(lr_model, background_data)` sebagai gantinya.

Untuk plot global, kita ambil **sample** dari validation set agar perhitungan tetap cepat walau datasetnya besar; untuk penjelasan per-nasabah (lokal), SHAP dihitung on-demand per baris.

In [ ]:
# ── BANGUN EXPLAINER & HITUNG SHAP VALUES ────────────────────────────────────
import json

SHAP_MODEL       = xgb_model   # ganti ke lgb_model / rf_model untuk explain model lain
SHAP_SAMPLE_SIZE = 1000        # sampling agar plot global tetap cepat di dataset besar

def _fix_xgboost_base_score_for_shap(model):
    """
    Workaround bug kompatibilitas SHAP <-> XGBoost >= 3.0 (belum ada fix resmi
    per Maret 2026, lihat https://github.com/shap/shap/issues/4288).
    XGBoost >= 3.0 menyimpan `base_score` sebagai string list, mis. '[5E-1]',
    sedangkan shap.TreeExplainer mengharapkan float biasa -> ValueError.
    Fungsi ini membetulkan format base_score di config booster secara in-place.
    Aman dipanggil berkali-kali / pada model non-XGBoost (langsung di-skip).
    """
    if not hasattr(model, "get_booster"):
        return model
    booster = model.get_booster()
    config  = json.loads(booster.save_config())
    lmp     = config["learner"]["learner_model_param"]
    bs      = lmp["base_score"]
    if isinstance(bs, list):
        bs = bs[0]
    if isinstance(bs, str) and bs.strip().startswith("["):
        bs = bs.strip("[]").split(",")[0]
    lmp["base_score"] = str(float(bs))
    booster.load_config(json.dumps(config))
    return model

SHAP_MODEL = _fix_xgboost_base_score_for_shap(SHAP_MODEL)

rng        = np.random.RandomState(42)
sample_idx = rng.choice(X_val.index, size=min(SHAP_SAMPLE_SIZE, len(X_val)), replace=False)
X_shap     = X_val.loc[sample_idx]

explainer   = shap.TreeExplainer(SHAP_MODEL)
shap_values = explainer.shap_values(X_shap)

# Beberapa versi/model classifier biner mengembalikan list [kontribusi_kelas0, kontribusi_kelas1]
# — kita ambil kontribusi terhadap kelas Default (1)
if isinstance(shap_values, list):
    shap_values    = shap_values[1]
    expected_value = explainer.expected_value[1]
else:
    expected_value = explainer.expected_value
    if isinstance(expected_value, (list, np.ndarray)):
        expected_value = expected_value[1]

print(f"Model yang dijelaskan   : {SHAP_MODEL.__class__.__name__}")
print(f"SHAP values dihitung    : {X_shap.shape[0]} sampel × {X_shap.shape[1]} fitur")
print(f"Expected value (base, log-odds): {expected_value:.4f}")

### 11.5.2 Global Feature Importance — SHAP Summary Plot

**Beeswarm plot**: setiap titik = satu nasabah. Posisi horizontal = besar & arah kontribusi terhadap risiko default (kanan = menaikkan risiko, kiri = menurunkan). Warna = nilai fitur (merah = tinggi, biru = rendah).

In [ ]:
# ── SUMMARY PLOT (BEESWARM) — KONTRIBUSI & ARAH PENGARUH TIAP FITUR ──────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='dot', show=False, max_display=20)
plt.title('SHAP Summary Plot — Pengaruh Fitur terhadap Risiko Default', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/shap_summary_beeswarm.png', dpi=100, bbox_inches='tight')
plt.show()

# ── BAR PLOT — RANKING GLOBAL BERDASARKAN RATA-RATA |SHAP VALUE| ─────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=20)
plt.title('SHAP Feature Importance — Rata-rata |SHAP Value|', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/shap_summary_bar.png', dpi=100, bbox_inches='tight')
plt.show()

### 11.5.3 Dependence Plot — Hubungan Nilai Fitur vs Dampaknya ke Prediksi

Menunjukkan bagaimana nilai suatu fitur (sumbu-x) berhubungan dengan kontribusi SHAP-nya (sumbu-y), termasuk kemungkinan interaksi dengan fitur lain (warna).

In [ ]:
# ── DEPENDENCE PLOT UNTUK TOP 3 FITUR PALING BERPENGARUH ─────────────────────
top_features = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=X_shap.columns)
    .sort_values(ascending=False)
    .head(3)
    .index.tolist()
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top_features):
    shap.dependence_plot(feat, shap_values, X_shap, ax=ax, show=False, interaction_index='auto')
    ax.set_title(f'SHAP Dependence — {feat}')
plt.tight_layout()
plt.savefig('/tmp/shap_dependence_top3.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Top 3 fitur paling berpengaruh (rata-rata |SHAP|): {top_features}")

### 11.5.4 Sample Penerapan — Penjelasan Prediksi per Nasabah (Local Explainability)

Ini contoh penerapan praktis: fungsi `explain_applicant()` di bawah bisa dipanggil untuk **satu nasabah** (mis. saat petugas kredit/relationship manager butuh tahu *alasan* di balik skor, atau saat sistem perlu menyertakan *reason codes* pada penolakan kredit). Fungsi ini bisa langsung diadaptasi ke endpoint scoring real-time (mis. SageMaker endpoint / Lambda) — cukup load `explainer` & model yang sudah disimpan, lalu panggil dengan data nasabah baru.

In [ ]:
def explain_applicant(row_idx, model=SHAP_MODEL, explainer=explainer, X_ref=X_val, top_n=10):
    """
    Hasilkan penjelasan SHAP untuk satu nasabah (diidentifikasi lewat index baris di X_ref).
    Mengembalikan DataFrame kontribusi fitur (diurutkan dari paling berpengaruh) dan
    menampilkan waterfall plot.

    Cocok dipakai di pipeline produksi (mis. di belakang endpoint scoring) untuk
    menyertakan 'alasan keputusan' di samping skor probabilitas.
    """
    x_row = X_ref.loc[[row_idx]]
    proba = model.predict_proba(x_row)[0, 1]

    sv = explainer.shap_values(x_row)
    if isinstance(sv, list):
        sv = sv[1]
    sv = sv[0]

    contrib = pd.DataFrame({
        'fitur'      : x_row.columns,
        'nilai_fitur': x_row.iloc[0].values,
        'shap_value' : sv,
    }).sort_values('shap_value', key=np.abs, ascending=False)

    print(f"Index nasabah (X_val)  : {row_idx}")
    print(f"Probabilitas Default   : {proba:.2%}")
    print(f"\nTop {top_n} fitur paling berpengaruh terhadap skor nasabah ini:")
    print(contrib.head(top_n).to_string(index=False))

    explanation = shap.Explanation(
        values=sv,
        base_values=expected_value,
        data=x_row.iloc[0].values,
        feature_names=x_row.columns.tolist()
    )
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(explanation, max_display=top_n, show=False)
    plt.title(f'Penjelasan Prediksi — Index {row_idx} (P(Default)={proba:.1%})')
    plt.tight_layout()
    plt.show()

    return contrib


# ── CONTOH PENERAPAN: BANDINGKAN NASABAH BERISIKO TINGGI vs RENDAH ───────────
val_proba_series = pd.Series(SHAP_MODEL.predict_proba(X_val)[:, 1], index=X_val.index)

high_risk_idx = val_proba_series.idxmax()   # nasabah dengan probabilitas default tertinggi
low_risk_idx  = val_proba_series.idxmin()   # nasabah dengan probabilitas default terendah

print("=" * 70)
print("CONTOH 1 — NASABAH BERISIKO TINGGI")
print("=" * 70)
contrib_high = explain_applicant(high_risk_idx)

print("\n" + "=" * 70)
print("CONTOH 2 — NASABAH BERISIKO RENDAH")
print("=" * 70)
contrib_low = explain_applicant(low_risk_idx)

### 11.5.5 Catatan Penerapan di Produksi

- **Plot global** (summary/dependence) cukup dihitung dari sample representatif (`SHAP_SAMPLE_SIZE`) — tidak perlu seluruh dataset.
- **Penjelasan per-nasabah** (`explain_applicant`) dihitung on-demand saat skor diminta, bukan di-precompute untuk semua nasabah.
- `TreeExplainer` eksak & cepat untuk model tree-based. Untuk Logistic Regression pakai `shap.LinearExplainer(lr_model, background_data)`; untuk model arbitrary (mis. ensemble custom) bisa pakai `shap.KernelExplainer` tapi jauh lebih lambat.
- Simpan `explainer` sebagai artifact bersama model (mis. `joblib.dump(explainer, f'{MODEL_DIR}/shap_explainer.pkl')` di Bagian 13) agar service scoring real-time bisa langsung generate *reason codes* tanpa rebuild explainer dari nol.
- Top-N fitur dari `contrib` (lihat fungsi `explain_applicant`) bisa langsung dipetakan ke kalimat alasan yang dipahami nasabah (mis. "rasio utang terhadap pendapatan yang tinggi" untuk `debt_to_income`), sering jadi requirement regulasi credit scoring (adverse action notice / *reason codes*).

## 12. Generate Prediksi Test & Simpan ke S3

In [ ]:
# ── ENCODE TEST SET ───────────────────────────────────────────────────────────
X_test = df_test[[c for c in feature_cols if c in df_test.columns]].copy()

for col in CAT_COLS:
    if col in X_test.columns:
        le = label_encoders[col]
        X_test[col] = X_test[col].astype(str).fillna('__missing__')
        # Handle unseen labels
        unseen_mask = ~X_test[col].isin(le.classes_)
        X_test.loc[unseen_mask, col] = '__missing__'
        if '__missing__' not in le.classes_:
            le.classes_ = np.append(le.classes_, '__missing__')
        X_test[col] = le.transform(X_test[col])

# Tambahkan kolom yang mungkin hilang
for col in feature_cols:
    if col not in X_test.columns:
        X_test[col] = np.nan
X_test = X_test[feature_cols]

# ── PREDIKSI DENGAN SEMUA MODEL ───────────────────────────────────────────────
test_lr  = lr_model.predict_proba(scaler.transform(imputer.transform(X_test)))[:, 1]
test_rf  = rf_model.predict_proba(imputer_rf.transform(X_test))[:, 1]
test_xgb = xgb_model.predict_proba(X_test)[:, 1]
test_lgb = lgb_model.predict_proba(X_test)[:, 1]

# ── ENSEMBLE (rata-rata probabilitas semua model) ─────────────────────────────
test_ensemble = (test_lr + test_rf + test_xgb + test_lgb) / 4

# ── BUAT SUBMISSION DATAFRAME ─────────────────────────────────────────────────
submission = pd.DataFrame({
    'SK_ID_CURR'             : df_test['SK_ID_CURR'],
    'TARGET_lr'              : test_lr,
    'TARGET_rf'              : test_rf,
    'TARGET_xgb'             : test_xgb,
    'TARGET_lgb'             : test_lgb,
    'TARGET_ensemble'        : test_ensemble,
})

print(f"Submission shape: {submission.shape}")
print(submission.head(10))

In [ ]:
# ── SIMPAN KE S3 VIA PYARROW (hindari konflik extension type) ─────────────────
import pyarrow as pa
import pyarrow.parquet as pq
import io

s3_client = boto3.client('s3')
OUTPUT_KEY_PQ  = "04-predictions/credit_risk_predictions.parquet"
OUTPUT_KEY_CSV = "04-predictions/credit_risk_predictions.csv"

# Simpan parquet: DataFrame → pyarrow Table → buffer → S3
buf = io.BytesIO()
pq.write_table(pa.Table.from_pandas(submission, preserve_index=False), buf)
buf.seek(0)
s3_client.upload_fileobj(buf, BUCKET, OUTPUT_KEY_PQ)

# Simpan CSV versi ringkas
csv_buf = io.StringIO()
submission[['SK_ID_CURR', 'TARGET_lgb', 'TARGET_ensemble']].to_csv(csv_buf, index=False)
s3_client.put_object(Body=csv_buf.getvalue(), Bucket=BUCKET, Key=OUTPUT_KEY_CSV)

print(f"[SUCCESS] Prediksi tersimpan di:")
print(f"  s3://{BUCKET}/{OUTPUT_KEY_PQ}")
print(f"  s3://{BUCKET}/{OUTPUT_KEY_CSV}")

print("\nDistribusi probabilitas default (ensemble):")
print(submission['TARGET_ensemble'].describe())

## 13. Simpan Model ke S3 (untuk Deployment)

In [ ]:
import joblib
import tarfile
import os

MODEL_DIR = '/tmp/credit_risk_models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Simpan semua model
joblib.dump(lr_model,  f'{MODEL_DIR}/lr_model.pkl')
joblib.dump(rf_model,  f'{MODEL_DIR}/rf_model.pkl')
joblib.dump(xgb_model, f'{MODEL_DIR}/xgb_model.pkl')
joblib.dump(lgb_model, f'{MODEL_DIR}/lgb_model.pkl')

# Simpan preprocessing artifacts
joblib.dump(imputer,         f'{MODEL_DIR}/imputer_lr.pkl')
joblib.dump(scaler,          f'{MODEL_DIR}/scaler_lr.pkl')
joblib.dump(imputer_rf,      f'{MODEL_DIR}/imputer_rf.pkl')
joblib.dump(label_encoders,  f'{MODEL_DIR}/label_encoders.pkl')
joblib.dump(feature_cols,    f'{MODEL_DIR}/feature_cols.pkl')

# Simpan metadata
import json
metadata = {
    'model_versions': {
        'logistic_regression': {'roc_auc': lr_auc,  'pr_auc': lr_aucpr},
        'random_forest'      : {'roc_auc': rf_auc,  'pr_auc': rf_aucpr},
        'xgboost'            : {'roc_auc': xgb_auc, 'pr_auc': xgb_aucpr},
        'lightgbm'           : {'roc_auc': lgb_auc, 'pr_auc': lgb_aucpr},
    },
    'feature_cols': feature_cols,
    'cat_cols'    : CAT_COLS,
    'num_cols'    : NUM_COLS,
    'train_rows'  : len(df_train),
    'val_rows'    : len(X_val),
    'default_rate': float(y.mean()),
}
with open(f'{MODEL_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# ── TAR & UPLOAD KE S3 ────────────────────────────────────────────────────────
TAR_PATH = '/tmp/credit_risk_models.tar.gz'
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    tar.add(MODEL_DIR, arcname='models')

s3 = boto3.client('s3')
s3.upload_file(TAR_PATH, BUCKET, '04-predictions/model_artifacts/credit_risk_models.tar.gz')

print("[SUCCESS] Model artifacts tersimpan di:")
print(f"  s3://{BUCKET}/04-predictions/model_artifacts/credit_risk_models.tar.gz")

## 14. Ringkasan Akhir

In [ ]:
print("=" * 60)
print("         RINGKASAN HASIL CREDIT RISK MODELING")
print("=" * 60)
print(f"  Dataset    : {len(df_train):,} nasabah (train) | {len(df_test):,} (test)")
print(f"  Fitur      : {len(feature_cols)} kolom (numerik + kategori)")
print(f"  Default rate: {y.mean():.2%}")
print()
print("  ROC-AUC Performance:")
print(f"  ├─ Logistic Regression (baseline) : {lr_auc:.4f}")
print(f"  ├─ Random Forest                  : {rf_auc:.4f}")
print(f"  ├─ XGBoost                        : {xgb_auc:.4f}")
print(f"  └─ LightGBM                       : {lgb_auc:.4f}")
print()
best_model = max([('Logistic Regression', lr_auc), ('Random Forest', rf_auc),
                   ('XGBoost', xgb_auc), ('LightGBM', lgb_auc)], key=lambda x: x[1])
print(f"  ★ Best model : {best_model[0]} (AUC={best_model[1]:.4f})")
print("=" * 60)